# 01 – Đánh giá CV Parser (Module A)

P/R/F1 (micro) của từng tầng trích kỹ năng trên 50 CV gán nhãn tay, và MAE số năm kinh nghiệm.

Đầu vào: `backend/seed/cv_files/*` (PDF/DOCX/TXT) và `backend/seed/labels/labels_cv_skills.csv` (`cv_file,skills[,years_exp]`).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

# Có thể đổi bằng biến môi trường khi chạy thử trên dữ liệu khác
DATA = Path(os.getenv("TH3_DATA_DIR", ROOT / "backend/seed/data"))
LABELS = Path(os.getenv("TH3_LABELS_DIR", ROOT / "backend/seed/labels"))
OUTDIR = Path(os.getenv("TH3_OUT_DIR", ROOT / "docs/report/tables"))
MODELS_OUT = Path(os.getenv("TH3_MODELS_OUT", ROOT / "backend/models_store"))
OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
from app.ai import datasets, evaluate
from app.ai.embedder import get_embedder
from app.ai.skills import SkillIndex

CV_DIR = Path(os.getenv("TH3_CV_DIR", ROOT / "backend/seed/cv_files"))
skills_csv, aliases_csv = DATA / "skills.csv", DATA / "skill_aliases.csv"
index = SkillIndex.from_csv(skills_csv, aliases_csv) if skills_csv.exists() else SkillIndex.default()
print("Số kỹ năng trong từ điển:", len(index.names), "| số cụm khoá:", len(index.phrases))
gold, years = datasets.load_cv_labels(LABELS / "labels_cv_skills.csv")
print("Số CV có nhãn:", len(gold))

In [ ]:
table = evaluate.evaluate_parser(CV_DIR, gold, years, index=index, embedder=get_embedder())
table.round(4)

In [ ]:
print("MAE số năm kinh nghiệm:", table.attrs.get("years_mae"))
table.to_csv(OUTDIR / "parser_eval.csv", index=False)
table.round(4)